# 寒武纪 (688256.SH) 近一年交易数据分析

本 Notebook 演示通过 **Tushare Pro** 获取 A 股日线数据，并进行可视化分析的全流程。

- **标的**: 寒武纪 688256.SH
- **区间**: 近一年（~242 个交易日）
- **工具**: requests, pandas, numpy, matplotlib, mplfinance

## 1. 环境准备

In [ ]:
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import mplfinance as mpf
from datetime import datetime, timedelta

# 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('环境就绪')

## 2. 通过 Tushare 获取数据

调用 `daily` 接口，获取 688256.SH 近一年的日线数据（开高低收 + 成交量）。

In [ ]:
# Tushare 配置
TOKEN = 'ef3f0720e87119a6cf39ae00eabfe51ee74837326edfe949bd2b1006'
API_URL = 'https://api.tushare.pro/'

ts_code = '688256.SH'

# 计算日期范围：一年前 ～ 今天
end_date = datetime.now().strftime('%Y%m%d')
start_date = (datetime.now() - timedelta(days=370)).strftime('%Y%m%d')

print(f'查询区间: {start_date} ~ {end_date}')
print(f'标    的: {ts_code}')

# 调用 Tushare daily 接口
payload = {
    'api_name': 'daily',
    'token': TOKEN,
    'params': {
        'ts_code': ts_code,
        'start_date': start_date,
        'end_date': end_date
    }
}

resp = requests.post(API_URL, json=payload)
data = resp.json()

if data['code'] != 0:
    print(f'API 错误: {data["msg"]}')
else:
    print(f'获取成功: {len(data["data"]["items"])} 条记录')
    print(f'字段: {data["data"]["fields"]}')

## 3. 数据预处理

将 JSON 转为 Pandas DataFrame，日期倒序改为正序，计算均线。

In [ ]:
# 构建 DataFrame
fields = data['data']['fields']
items = data['data']['items']
df = pd.DataFrame(items, columns=fields)

# 类型转换
df['trade_date'] = pd.to_datetime(df['trade_date'])
for col in ['open', 'high', 'low', 'close', 'pre_close', 'change', 
             'pct_chg', 'vol', 'amount']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 按日期升序排列
df = df.sort_values('trade_date', ascending=True).reset_index(drop=True)

# 计算均线
df['MA5']  = df['close'].rolling(window=5).mean()
df['MA10'] = df['close'].rolling(window=10).mean()
df['MA20'] = df['close'].rolling(window=20).mean()
df['MA60'] = df['close'].rolling(window=60).mean()

print(f'数据范围: {df["trade_date"].min().date()} ~ {df["trade_date"].max().date()}')
print(f'交易天数: {len(df)}')
print(f'收盘价: {df["close"].min():.2f} ~ {df["close"].max():.2f}')
df.head(10)

## 4. K 线图 + 均线 + 成交量

使用 **mplfinance** 绘制专业级 K 线图，A 股配色（涨红跌绿），叠加 MA5/MA10/MA20/MA60 四根均线。

In [ ]:
# 准备 mplfinance 格式数据
df_k = df[['trade_date', 'open', 'high', 'low', 'close', 'vol']].copy()
df_k = df_k.set_index('trade_date')
df_k.index.name = 'Date'
df_k.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

# 准备均线叠加
apds = mpf.make_addplot(df['MA5'].values,  panel=0, color='#2196F3', width=0.8, label='MA5')
apds.append(mpf.make_addplot(df['MA10'].values, panel=0, color='#FF9800', width=0.8, label='MA10'))
apds.append(mpf.make_addplot(df['MA20'].values, panel=0, color='#F44336', width=0.8, label='MA20'))
apds.append(mpf.make_addplot(df['MA60'].values, panel=0, color='#4CAF50', width=1.0, label='MA60'))

# A 股配色: 涨红跌绿
mc = mpf.make_marketcolors(
    up='#ef5350',    # 涨 = 红
    down='#26a69a',  # 跌 = 绿
    edge='inherit',
    volume={'up': '#ef5350', 'down': '#26a69a'},
    alpha=0.85
)
s = mpf.make_mpf_style(marketcolors=mc, gridstyle='--', gridcolor='#e0e0e0')

fig, axes = mpf.plot(
    df_k,
    type='candle',
    style=s,
    title='\n寒武纪 688256.SH 日K线图\n',
    ylabel='价格 (元)',
    ylabel_lower='成交量',
    volume=True,
    addplot=apds,
    figsize=(16, 9),
    returnfig=True,
    datetime_format='%Y-%m',
    xrotation=45,
    tight_layout=True
)

# 添加图例
axes[0].legend(['MA5', 'MA10', 'MA20', 'MA60'], loc='upper left')
plt.show()

## 5. 收盘价走势图

用 matplotlib 绘制收盘价 + 四根均线的叠加曲线。

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

# 背景填充
ax.fill_between(df['trade_date'], df['close'], alpha=0.04, color='#1565C0')

# 收盘价
ax.plot(df['trade_date'], df['close'], 
        color='#1565C0', linewidth=1.5, label='收盘价', zorder=3)

# 均线
ax.plot(df['trade_date'], df['MA5'],  color='#2196F3', linewidth=0.8, label='MA5')
ax.plot(df['trade_date'], df['MA10'], color='#FF9800', linewidth=0.8, label='MA10')
ax.plot(df['trade_date'], df['MA20'], color='#F44336', linewidth=0.8, label='MA20')
ax.plot(df['trade_date'], df['MA60'], color='#4CAF50', linewidth=1.0, label='MA60')

# 标注最高/最低点
max_idx = df['close'].idxmax()
min_idx = df['close'].idxmin()
ax.annotate(f"最高 ¥{df.loc[max_idx, 'close']:.2f}",
            xy=(df.loc[max_idx, 'trade_date'], df.loc[max_idx, 'close']),
            xytext=(0, 15), textcoords='offset points',
            fontsize=10, color='#F44336', fontweight='bold',
            ha='center',
            arrowprops=dict(arrowstyle='->', color='#F44336'))
ax.annotate(f"最低 ¥{df.loc[min_idx, 'close']:.2f}",
            xy=(df.loc[min_idx, 'trade_date'], df.loc[min_idx, 'close']),
            xytext=(0, -20), textcoords='offset points',
            fontsize=10, color='#26a69a', fontweight='bold',
            ha='center',
            arrowprops=dict(arrowstyle='->', color='#26a69a'))

# 格式化
ax.set_title('寒武纪 688256.SH 收盘价走势', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('日期', fontsize=12)
ax.set_ylabel('收盘价 (元)', fontsize=12)
ax.legend(loc='upper left', framealpha=0.9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.xticks(rotation=45)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_facecolor('#FAFAFA')

plt.tight_layout()
plt.show()

## 6. 涨跌分布分析

In [ ]:
df['is_up'] = df['change'] >= 0

up_days = df['is_up'].sum()
down_days = (~df['is_up']).sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 饼图: 涨跌天数
colors = ['#ef5350', '#26a69a']
labels = [f'上涨  {up_days}天', f'下跌  {down_days}天']
sizes = [up_days, down_days]
explode = (0.03, 0)
axes[0].pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
            shadow=False, startangle=90, textprops={'fontsize': 13})
axes[0].set_title('涨跌天数分布', fontsize=14, fontweight='bold')

# 直方图: 涨跌幅分布
axes[1].hist(df['pct_chg'], bins=30, color='#1565C0', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='#333', linestyle='--', linewidth=1, alpha=0.5)
axes[1].axvline(x=df['pct_chg'].mean(), color='#F44336', linestyle='-', linewidth=1.5,
                label=f"均值 {df['pct_chg'].mean():.2f}%")
axes[1].set_title('日涨跌幅分布', fontsize=14, fontweight='bold')
axes[1].set_xlabel('涨跌幅 (%)')
axes[1].set_ylabel('频次')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 统计汇总

In [ ]:
close_start = df.iloc[0]['close']
close_end   = df.iloc[-1]['close']
change_pct  = (close_end - close_start) / close_start * 100

print('=' * 50)
print('     寒武纪 688256.SH 数据摘要')
print('=' * 50)
print(f'  数据区间 : {df["trade_date"].min().date()}  ~  {df["trade_date"].max().date()}')
print(f'  交易天数 : {len(df)} 天')
print(f'  上涨天数 : {up_days} 天  ({up_days/len(df)*100:.1f}%)')
print(f'  下跌天数 : {down_days} 天  ({down_days/len(df)*100:.1f}%)')
print(f'  最低收盘 : ¥{df["close"].min():.2f}')
print(f'  最高收盘 : ¥{df["close"].max():.2f}')
print(f'  平均收盘 : ¥{df["close"].mean():.2f}')
print(f'  期初收盘 : ¥{close_start:.2f}')
print(f'  期末收盘 : ¥{close_end:.2f}')
print(f'  区间涨跌 : {"+" if change_pct >= 0 else ""}{change_pct:.2f}%')
print(f'  最大涨幅 : +{df["pct_chg"].max():.2f}%')
print(f'  最大跌幅 : {df["pct_chg"].min():.2f}%')
print(f'  日均成交量 : {df["vol"].mean():.0f} 手')
print('=' * 50)

## 8. 保存 CSV 数据

In [ ]:
csv_path = '688256_SH_daily.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'数据已保存: {csv_path}')
print(f'文件大小: {len(df)} 行 x {len(df.columns)} 列')

---

## 总结

本 Notebook 演示了从数据获取到可视化的完整流程：

1. **Tushare API** → `daily` 接口拉取 A 股日线
2. **数据清洗** → Pandas 类型转换、日期排序、均线计算
3. **K 线图** → mplfinance 专业蜡烛图 + 四根均线 + 成交量
4. **收盘价趋势** → matplotlib 叠加均线的折线图
5. **涨跌分布** → 饼图 + 直方图分析
6. **CSV 导出** → 含均线字段的完整数据文件

可直接替换 `ts_code` 分析任意沪深 A 股。